In [ ]:
!pip install "datasets<3.0.0" soundfile transformers accelerate torch resemblyzer librosa pandas scipy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 5.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... canceled


In [ ]:
import torch
import numpy as np
import pandas as pd
import soundfile as sf
import os
from pathlib import Path
from scipy.signal import resample

from transformers import VitsModel, AutoTokenizer
from resemblyzer import VoiceEncoder, preprocess_wav

# Global Config
NUM_SAMPLES = 15
MODEL_NAME = "facebook/mms-tts-urd-script_arabic"

# Load Models
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = VitsModel.from_pretrained(MODEL_NAME)
model.eval()
encoder = VoiceEncoder()

def run_evaluation(dataset_name, samples_list, text_key):
    base_dir = f"mushra_{dataset_name}"
    for sub in ["reference", "generated", "anchor"]:
        os.makedirs(f"{base_dir}/{sub}", exist_ok=True)

    results = []
    for i in range(min(NUM_SAMPLES, len(samples_list))):
        try:
            sample = samples_list[i]
            text = sample[text_key]
            ref_array = np.array(sample['audio']['array']).astype(np.float32)
            ref_sr = sample['audio']['sampling_rate']

            if ref_array.ndim > 1:
                ref_array = ref_array.mean(axis=1)

            # Paths
            ref_path = f"{base_dir}/reference/sample_{i}.wav"
            gen_path = f"{base_dir}/generated/sample_{i}.wav"
            anc_path = f"{base_dir}/anchor/sample_{i}.wav"

            # 1. Save Reference
            sf.write(ref_path, ref_array, ref_sr)

            # 2. Generate TTS
            inputs = tokenizer(text, return_tensors="pt")
            with torch.no_grad():
                output = model(**inputs)
            gen_speech = output.waveform[0].cpu().numpy()
            gen_sr = model.config.sampling_rate
            sf.write(gen_path, gen_speech, gen_sr)

            # 3. Create Anchor (8kHz resample cycle)
            target_len_8k = int(len(ref_array) * 8000 / ref_sr)
            downsampled = resample(ref_array, target_len_8k)
            upsampled = resample(downsampled, len(ref_array))
            sf.write(anc_path, upsampled, ref_sr)

            # 4. Resemblyzer Metrics
            wav_ref = preprocess_wav(Path(ref_path))
            wav_gen = preprocess_wav(Path(gen_path))

            # Split reference in half to compute self-similarity (consistency check)
            mid = len(wav_ref) // 2
            emb_ref_full = encoder.embed_utterance(wav_ref)
            emb_ref_a = encoder.embed_utterance(wav_ref[:mid])
            emb_ref_b = encoder.embed_utterance(wav_ref[mid:])
            emb_gen = encoder.embed_utterance(wav_gen)

            # Similarity calculations
            ref_gen_sim = np.dot(emb_ref_full, emb_gen) / (np.linalg.norm(emb_ref_full) * np.linalg.norm(emb_gen))
            ref_self_sim = np.dot(emb_ref_a, emb_ref_b) / (np.linalg.norm(emb_ref_a) * np.linalg.norm(emb_ref_b))

            results.append({
                "sample_id": i,
                "urdu_text": text,
                "audio_duration_sec": round(len(ref_array)/ref_sr, 2),
                "ref_gen_similarity": float(ref_gen_sim),
                "ref_self_similarity": float(ref_self_sim)
            })
            print(f"[{dataset_name.upper()}] Sample {i+1}/{NUM_SAMPLES}: sim={ref_gen_sim:.4f} | self_sim={ref_self_sim:.4f}")

        except Exception as e:
            print(f"Error in {dataset_name} sample {i}: {e}")
            continue

    # Mini Summary
    df_temp = pd.DataFrame(results)
    embedding_gap = df_temp['ref_self_similarity'].mean() - df_temp['ref_gen_similarity'].mean()
    print(f"\n--- {dataset_name.upper()} Summary ---")
    print(f"Mean Gen Sim: {df_temp['ref_gen_similarity'].mean():.4f} | Embedding Gap: {embedding_gap:.4f}")
    return results

In [ ]:
from datasets import load_dataset
# Use streaming=True to fetch samples instantly
ds_fleurs_stream = load_dataset("google/fleurs", "ur_pk", split="test", trust_remote_code=True, streaming=True)

# Take first 15 samples from the stream
fleurs_samples = []
for sample in ds_fleurs_stream:
    fleurs_samples.append(sample)
    if len(fleurs_samples) >= NUM_SAMPLES:
        break

results_fleurs = run_evaluation("fleurs", fleurs_samples, "transcription")

In [ ]:
ds_cv = load_dataset("khawajaaliarshad/common-voice-urdu-processed-expanded", split="train[:50]")
results_cv = run_evaluation("commonvoice", ds_cv.select(range(NUM_SAMPLES)), "sentence")

In [ ]:
from google.colab import userdata
try:
    hf_token = userdata.get('HF_TOKEN')
except:
    hf_token = None # Fallback if secret not set

ds_urdutts = load_dataset("muhammadsaadgondal/urdu-tts", split="train[:50]", token=hf_token)
results_urdutts = run_evaluation("urdutts", ds_urdutts.select(range(NUM_SAMPLES)), "text")

In [ ]:
# Target the 'Urdu' directory directly to avoid streaming irrelevant Kashmiri shards
ds_urduspeech_stream = load_dataset("humairawan/UrduSpeech", data_dir="Urdu", split="train", token=hf_token, streaming=True)

# Take first 15 samples from the Urdu-specific stream
urduspeech_samples = []
for sample in ds_urduspeech_stream:
    urduspeech_samples.append(sample)
    if len(urduspeech_samples) >= NUM_SAMPLES:
        break

results_urduspeech = run_evaluation("urduspeech", urduspeech_samples, "text")

In [ ]:
all_data = []
for name, res in [("fleurs", results_fleurs), ("commonvoice", results_cv),
                  ("urdutts", results_urdutts), ("urduspeech", results_urduspeech)]:
    for entry in res:
        entry['dataset'] = name
        all_data.append(entry)

df = pd.DataFrame(all_data)
df = df[['dataset', 'sample_id', 'urdu_text', 'audio_duration_sec', 'ref_gen_similarity', 'ref_self_similarity']]

# Summary Table
stats = df.groupby('dataset').agg({
    'ref_gen_similarity': 'mean',
    'ref_self_similarity': 'mean'
})

stats.columns = ['mean_gen_sim', 'mean_self_sim']
stats['embedding_gap'] = stats['mean_self_sim'] - stats['mean_gen_sim']

print("=== GLOBAL SUMMARY STATISTICS ===")
display(stats)

df.to_csv("urdu_tts_evaluation_all_datasets.csv", index=False, encoding='utf-8-sig')
print("\nFull results saved to urdu_tts_evaluation_all_datasets.csv")

In [ ]:
import zipfile
datasets = ["fleurs", "commonvoice", "urdutts", "urduspeech"]

with zipfile.ZipFile("mushra_all_datasets.zip", "w") as zf:
    for d_name in datasets:
        folder = f"mushra_{d_name}"
        for root, dirs, files in os.walk(folder):
            for file in files:
                zf.write(os.path.join(root, file))

print("Successfully created mushra_all_datasets.zip containing all audio samples.")